-------------------------------
-------------------------------
# Proyecto 2 - Minería de Datos (CC3074)
* Dulce Ambrosio - 231143
* Daniel Chet - 231177
* Javier Linares - 231135
* Cristian Túnchez - 231359

-------------------------------
-------------------------------

-------------------------------
**Semana 3:** Mejora de modelos + seleción

-------------------------------

-------------------------------
Importación de librerías y carga del dataset (Se realizó basandose en la semana 1 y semana 2)

-------------------------------

In [ ]:
# Importación de librerías base para manejo de datos y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suprimir advertencias para mantener la salida limpia
import warnings
warnings.filterwarnings('ignore')

# Importación de modelos y utilidades de scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, learning_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Configuración de visualización (tamaño de figura y estilo)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')


In [ ]:
# Carga y limpieza del dataset (reproduciendo el pipeline de Semanas 1 y 2)
# - Lee el CSV
# - Convierte la columna "Bare Nuclei" a numérico (si hubiera '?', se vuelve NaN)
# - Elimina registros con valores faltantes
# - Asegura el tipo de "Class" y elimina el identificador si existe

df = pd.read_csv('Datos.csv')

# Limpieza: Bare Nuclei puede tener '?' en algunas versiones del dataset
df['Bare Nuclei'] = pd.to_numeric(df['Bare Nuclei'], errors='coerce')
df.dropna(inplace=True)
df['Class'] = df['Class'].astype(int)

# Eliminar columna identificadora
if 'Sample code number' in df.columns:
    df.drop(columns=['Sample code number'], inplace=True)

# Resumen rápido del dataset resultante
print(f'Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas')
print(f'Distribución de clases:\n{df["Class"].value_counts()}')

In [ ]:
# Preparación de variables:
# - X: features (todas las columnas excepto Class)
# - y: etiqueta binaria (2 -> 0 Benigno, 4 -> 1 Maligno)
# - Escalado con StandardScaler (necesario para modelos sensibles a escala)
# - Split estratificado train/test para mantener proporciones de clase

X = df.drop(columns=['Class'])
y = df['Class'].map({2: 0, 4: 1})  # Binarizar: 0=Benigno, 1=Maligno

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba:        {X_test.shape[0]} muestras')

-------------------------------
**Cross - Validation (Validación Cruzada):** Se utiliza stratified K-fold con k=10 para estimar el rendimiendo de los modelos base de forma más confiable, evitando el sesgo de ua sola partición train/test

-------------------------------

In [ ]:
# Configuración de validación cruzada (Stratified K-Fold):
# - Mantiene proporción de clases en cada fold
# - Shuffle + random_state para reproducibilidad
#
# Evaluación de modelos "base" (sin tuning) usando métricas estándar.

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Modelos base (sin optimizar, para comparación)
modelos_base = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Árbol de Decisión':   DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42)
}

metricas_cv = {}

for nombre, modelo in modelos_base.items():
    # Cada métrica se estima en CV de forma independiente
    acc    = cross_val_score(modelo, X_scaled, y, cv=cv, scoring='accuracy')
    prec   = cross_val_score(modelo, X_scaled, y, cv=cv, scoring='precision')
    rec    = cross_val_score(modelo, X_scaled, y, cv=cv, scoring='recall')
    f1     = cross_val_score(modelo, X_scaled, y, cv=cv, scoring='f1')
    metricas_cv[nombre] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1
    }

    # Reporte: media ± desviación estándar
    print(f'\n{nombre}:')
    print(f'  Accuracy  : {acc.mean():.4f} ± {acc.std():.4f}')
    print(f'  Precision : {prec.mean():.4f} ± {prec.std():.4f}')
    print(f'  Recall    : {rec.mean():.4f} ± {rec.std():.4f}')
    print(f'  F1-Score  : {f1.mean():.4f} ± {f1.std():.4f}')

-------------------------------
**Evaluación de Modelos Optimizados en Test Set** 

-------------------------------

In [ ]:
# Evaluación en test set para los mejores estimadores encontrados en GridSearchCV
# Se calcula Accuracy, Precision, Recall, F1 y AUC-ROC (si el modelo soporta predict_proba).

def evaluar_modelo(nombre, modelo, X_train, y_train, X_test, y_test):
    # Entrenamiento final del estimador seleccionado
    modelo.fit(X_train, y_train)

    # Predicciones en el conjunto de prueba
    y_pred = modelo.predict(X_test)

    # Probabilidades (para AUC/ROC); algunos modelos podrían no implementarlo
    y_prob = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, 'predict_proba') else None

    # Métricas principales
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    # AUC-ROC requiere probabilidades
    auc  = roc_auc_score(y_test, y_prob) if y_prob is not None else None

    # Matriz de confusión para ver FN/FP explícitamente
    cm   = confusion_matrix(y_test, y_pred)

    # Impresión del reporte del modelo
    print(f'\n{'='*50}')
    print(f'{nombre} (optimizado)')
    print(f'{'='*50}')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall   : {rec:.4f}')
    print(f'  F1-Score : {f1:.4f}')
    if auc: print(f'  AUC-ROC  : {auc:.4f}')
    print(f'  Matriz de confusión: {cm.tolist()}')

    # Retornar todo lo necesario para gráficas/comparación posterior
    return {'nombre': nombre, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'auc': auc, 'cm': cm, 'y_prob': y_prob}

# Evaluación de cada mejor estimador (uno por familia de modelos)
res_lr = evaluar_modelo('Regresión Logística', grid_lr.best_estimator_, X_train, y_train, X_test, y_test)
res_dt = evaluar_modelo('Árbol de Decisión',   grid_dt.best_estimator_, X_train, y_train, X_test, y_test)
res_rf = evaluar_modelo('Random Forest',       grid_rf.best_estimator_, X_train, y_train, X_test, y_test)

# Lista unificada para iterar en visualizaciones
resultados = [res_lr, res_dt, res_rf]

In [ ]:
# Matrices de confusión – modelos optimizados
# Visualiza TP/TN/FP/FN para cada modelo en el conjunto de prueba.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titulos = ['Regresión Logística\n(Optimizada)', 'Árbol de Decisión\n(Optimizado)', 'Random Forest\n(Optimizado)']

for ax, res, titulo in zip(axes, resultados, titulos):
    # Heatmap para lectura más clara
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benigno', 'Maligno'],
                yticklabels=['Benigno', 'Maligno'], ax=ax,
                linewidths=0.5, linecolor='gray')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.suptitle('Matrices de Confusión – Modelos Optimizados', fontsize=14, fontweight='bold')
plt.tight_layout()

-------------------------------
**Curvas ROC** 

-------------------------------

In [ ]:
# Curvas ROC (Receiver Operating Characteristic)
# - Se grafican TPR vs FPR para cada modelo optimizado
# - El AUC resume el área bajo la curva (mayor es mejor)

fig, ax = plt.subplots(figsize=(9, 7))

colores_roc = ['#2196F3', '#FF9800', '#4CAF50']
for res, color in zip(resultados, colores_roc):
    # Solo se puede trazar ROC si existen probabilidades
    if res['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
        ax.plot(fpr, tpr, color=color, lw=2.5,
                label=f"{res['nombre']} (AUC = {res['auc']:.4f})")

# Línea base: clasificador aleatorio
ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Clasificador aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC – Modelos Optimizados', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()

-------------------------------
**Comparación Formal: Base vs Optimizado** 

-------------------------------

In [ ]:
# Comparación formal: Base (Semana 2) vs Optimizado (Semana 3)
# Se reportan métricas por modelo y el delta (Optimizado - Base).

# Resultados base (Semana 2) para comparación directa
base = {
    'Regresión Logística': {'acc': 0.956, 'prec': 0.981, 'rec': 0.914, 'f1': 0.946},
    'Árbol de Decisión':   {'acc': 0.934, 'prec': 0.962, 'rec': 0.879, 'f1': 0.919},
    'Random Forest':       {'acc': 0.949, 'prec': 0.981, 'rec': 0.897, 'f1': 0.937},
}

print('\n')
print(f'{"Modelo":<25} {"Métrica":<12} {"Base":>8} {"Optimizado":>12} {"Δ":>8}')
print('-'*70)

for res in resultados:
    nombre = res['nombre']
    # Iteración por métricas para no repetir código
    for met_key, met_label in [("acc","Accuracy"),("prec","Precision"),("rec","Recall"),("f1","F1-Score")]:
        b = base[nombre][met_key]
        o = res[met_key]
        delta = o - b
        signo = '+' if delta >= 0 else ''
        print(f'{nombre:<25} {met_label:<12} {b:>8.4f} {o:>12.4f} {signo+f"{delta:.4f}":>8}')
    print()